# Train BandNet Mobile1D XL (~25 MB FP32) in Google Colab

Choose **Runtime → Change runtime type → T4 GPU**, then **Run all**. This is a new model trained **from scratch**, not a resume of the smaller 1.4 MiB Mobile1D model. Run it in a **different Colab session** from the existing 120-epoch run.

### Capacity and inputs
- **6,333,985 parameters** versus 369,116 in the original Mobile1D (about **17.2×** as many); estimated **24.16 MiB FP32** weights, not an actual `.tflite` file size.
- `base_channels=48`, `d_model=568`, TCN=3, cross-attention reader layers=2; RGB input stays **176×512** and 256 dense bins.
- **120 epochs**, batch size **8** for a 15 GiB T4 GPU, warmup 5 epochs, one cosine schedule with initial LR 0.00025, dropout 0.15.
- No change to resistor preprocessing, annotation splits, decoding outputs or deployment input shape. Improved accuracy is **not guaranteed**.

### Files and caching
- XL checkpoints and exports: `MyDrive/resistor_model/r1-mobile1d-xl-colab/`.
- Original checkpoints and model: `MyDrive/resistor_model/r1-mobile1d-colab/` — **untouched**.
- The new run reuses the existing verified prepared-dataset ZIP and release archives in the original folder; it **does not resume** the small-model weights because tensor shapes are different.
- Automatically logs every epoch, `last.pt`, `best.pt`, and numbered checkpoint snapshots. A compact results ZIP excludes recovery snapshots but keeps them on Drive. Automatic browser download is disabled for XL because checkpoints can be large.
- The XL model is expected to require substantially more training and mobile inference time. Benchmark actual devices and check Google Play services LiteRT compatibility before shipping.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

branch = 'feat/mobile1d-colab-progress-drive-cache'
os.environ['MOBILE1D_GIT_REF'] = branch
repo = Path('/content/mobile1d_xl_script_source')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run([
    'git', 'clone', '--depth', '1', '--single-branch', '--branch', branch,
    'https://github.com/Persie0/resistor_model.git', str(repo),
], check=True)
script = repo / 'colab' / 'train_rres_mobile1d_xl_colab.py'
base_script = repo / 'colab' / 'train_rres_mobile1d_colab.py'
assert script.is_file() and base_script.is_file()
assert 'XL_FEATURES = 568' in script.read_text()
print('[XL notebook] using commit:', subprocess.check_output([
    'git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'
], text=True).strip())
source = script.read_bytes()
exec(compile(source, str(script), 'exec'), {'__name__': '__main__', '__file__': str(script)})
